# Pelajaran 11 - Protokol Ejen-ke-Ejen (A2A)


## Persediaan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Apakah Protokol A2A?

Protokol **Agen-ke-Agen (A2A)** ialah satu standard terbuka yang membolehkan ejen AI berkomunikasi, menemui satu sama lain, dan bekerjasama — walaupun apabila mereka dibina pada rangka kerja yang berbeza atau dihoskan oleh perkhidmatan yang berbeza.

Key concepts:

- **Penemuan** – Ejen menerbitkan *Kad Ejen* yang menerangkan kebolehan mereka, menjadikannya mudah bagi ejen lain (atau pengatur) untuk mencari pakar yang sesuai untuk sesuatu tugasan.
- **Penghantaran Mesej** – Ejen bertukar mesej berstruktur melalui protokol umum, jadi permintaan daripada satu ejen boleh difahami dan dipenuhi oleh ejen lain tanpa mengira pelaksanaan dalaman mereka.
- **Kitaran Hayat Tugasan** – A2A mentakrifkan status seperti *submitted*, *working*, *completed*, dan *failed*, memberi pengatur (orchestrator) keterlihatan penuh ke atas bagaimana tugasan yang didelegasikan sedang berjalan.

Dalam pelajaran ini kami mensimulasikan kerjasama gaya A2A dengan menyambungkan tiga ejen pelancongan khusus ke dalam aliran kerja di mana setiap ejen menyumbangkan kepakaran mereka dan menyerahkan hasil kepada ejen seterusnya.


## Mencipta Ejen Pelancongan Khusus


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Kerjasama Berbilang Ejen melalui Aliran Kerja

Kami menghubungkan ketiga-tiga ejen ke dalam aliran kerja berurutan yang mencerminkan pemesejan A2A:

1. **CurrencyExchangeAgent** menerima permintaan pengguna dan menghasilkan panduan mata wang.
2. **ActivityPlannerAgent** menerima konteks yang diperkaya dan menambah cadangan aktiviti.
3. **TravelManagerAgent** mensintesis kedua-dua input menjadi ringkasan perjalanan akhir.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Memahami A2A dalam Persekitaran Pengeluaran

Dalam persekitaran pengeluaran protokol A2A membuka senario rentas-perkhidmatan yang berkuasa:

| Capability | Description |
|---|---|
| **Interoperabiliti merentas rangka kerja** | Ejen yang dibina dengan satu rangka kerja boleh mendelegasikan tugas kepada ejen yang dibina dengan mana-mana rangka kerja lain yang mematuhi A2A, membolehkan interoperabiliti sebenar antara organisasi. |
| **Sempadan perkhidmatan** | Ejen boleh wujud dalam mikroperkhidmatan yang berasingan, rantau awan, atau malah organisasi yang berbeza sambil terus berkolaborasi dengan lancar. |
| **Penemuan dinamik** | Orkestrator boleh meminta daftar Kad Ejen semasa runtime untuk mencari pakar yang paling sesuai bagi sub-tugas tertentu. |
| **Penstriman & pemberitahuan tolak** | A2A menyokong Server-Sent Events (SSE) untuk kemas kini kemajuan masa nyata dan pemberitahuan tolak untuk tugas yang berjalan lama. |

Aliran kerja yang kami bina di atas adalah versi ringkas, dalam-proses bagi corak ini. Dalam penempatan sebenar setiap ejen akan mendedahkan endpoint HTTP, menerbitkan Kad Ejen, dan berkomunikasi melalui protokol A2A JSON-RPC.


## Ringkasan

Dalam pelajaran ini anda telah mempelajari:

1. **Apa itu protokol A2A** — satu piawaian terbuka untuk penemuan agen-ke-agen, pemesejan,
   dan pengurusan tugas.
2. **Cara untuk mencipta agen khusus** — seorang agen Pertukaran Mata Wang, seorang agen Perancang Aktiviti,
   dan seorang pengorkestrasi Pengurus Perjalanan.
3. **Cara menghubungkan agen ke dalam aliran kerja** — menggunakan `WorkflowBuilder` untuk memodelkan sequential
   message passing antara agen.
4. **Bagaimana A2A berfungsi dalam pengeluaran** — membolehkan kerjasama lintas-rangka, lintas-perkhidmatan
   dengan penemuan dinamik dan kemas kini penstriman.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Penafian:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI Co-op Translator (https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi ralat atau ketidaktepatan. Dokumen asal dalam bahasa asalnya hendaklah dianggap sebagai sumber rujukan yang sahih. Untuk maklumat penting, disyorkan mendapatkan terjemahan profesional oleh penterjemah manusia. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsiran yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
